# LeetCode 3973 — Distinct Gate Paths to LCA bằng Python

> **Bài Premium: đề bài viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.** Tên phương thức dưới đây (`distinctGatePaths`) và thứ tự tham số `(n, parent, gates, queries)` được suy ra từ đề (LeetCode không cung cấp skeleton công khai cho bài Premium) — cần đối chiếu lại với trình soạn thảo thật.

- Link: https://leetcode.com/problems/distinct-gate-paths-to-lca/
- Độ khó: **Hard**
- Chủ đề: Cây, LCA (tổ tiên chung gần nhất), nhân lên (binary lifting), DP đếm đường đi, ma trận `2×2`, modulo, XOR

Skeleton (suy luận, cần đối chiếu):

```python
class Solution:
    def distinctGatePaths(self, n: int, parent: list[int], gates: list[list[int]], queries: list[list[int]]) -> int:
```

## 1. Giới thiệu & mục tiêu học

Có một cây gốc `0`. Mỗi nút có ba loại cổng: đỏ, xanh, trắng (với số lượng cho trước). Hai người Alice và Bob, mỗi người cầm một thẻ màu (đỏ hoặc xanh), cùng leo **lên** tới LCA của hai nút xuất phát. Mỗi bước leo từ `u` lên `parent[u]` phải chọn **một cổng cụ thể** tại `u`: cổng cùng màu thẻ giữ nguyên thẻ, cổng trắng đổi màu thẻ. Mỗi truy vấn cần đếm số cách đi của cả hai (nhân hai số cách lại), rồi lấy XOR tất cả đáp án.

Điểm mấu chốt: số cách đi của **một** người từ `a` lên tới tổ tiên `l` là **tích các ma trận `2×2`** dọc theo đường đi — mỗi nút đóng góp một ma trận "chuyển màu thẻ". Với `2·10^4` nút và `2·10^4` truy vấn, ta cần nhân dãy ma trận **dọc đường đi lên** thật nhanh bằng kỹ thuật nhân đôi (binary lifting).

Sau notebook này bạn sẽ:

1. Mô hình hóa "đếm đường đi qua một chuỗi lựa chọn có trạng thái" thành nhân ma trận.
2. Biết biểu diễn cây bằng mảng `parent`, tính độ sâu, tìm LCA.
3. Nắm binary lifting không chỉ cho "tổ tiên thứ `2^k`" mà cho cả **tích ma trận** trên đoạn `2^k` nút.
4. Viết 4 lời giải (liệt kê cổng, đi bộ từng nút, binary lifting, offline với segment tree) và đối chiếu bằng test ngẫu nhiên.
5. Phân tích độ phức tạp `O((n + q) log n)` và vì sao không dùng được "ma trận nghịch đảo" (tiền tố) ở đây.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho `n` nút, cây gốc `0` bằng `parent` (với `parent[0] = -1`). `gates[i] = [red_i, blue_i, white_i]` là số cổng đỏ, xanh, trắng ở nút `i`.

- Thẻ: `1` = đỏ, `0` = xanh.
- Khi rời nút `u` để lên `parent[u]`, ta chọn đúng **một cổng** (các cổng cùng loại được coi là **khác nhau**, đếm riêng):
  - thẻ đỏ: dùng cổng đỏ → giữ đỏ; hoặc cổng trắng → đổi thành xanh;
  - thẻ xanh: dùng cổng xanh → giữ xanh; hoặc cổng trắng → đổi thành đỏ;
  - không có cổng dùng được → không đi được (số cách bằng 0).
- Truy vấn `[aNode, aCard, bNode, bCard]`: Alice từ `aNode` với thẻ `aCard`, Bob từ `bNode` với thẻ `bCard`, mỗi người **độc lập** leo tới `LCA(aNode, bNode)`. Số cách của một người đã ở LCA là `1`. Số cách của truy vấn = (số cách Alice) × (số cách Bob), `mod 10^9 + 7`.
- Kết quả cuối: **XOR** của tất cả các số cách đó.

**Ví dụ 1:** `n = 3`, `parent = [-1,0,0]`, `gates = [[1,0,1],[0,1,1],[1,1,0]]`, `queries = [[1,0,2,0],[1,1,2,0],[1,0,2,1]]` → `1` (các số cách từng truy vấn: `2, 1, 2`, và `2 XOR 1 XOR 2 = 1`).

**Ví dụ 2:** `n = 3`, `parent = [-1,0,1]`, `gates = [[0,1,2],[1,0,1],[0,0,3]]`, `queries = [[2,0,1,0],[2,1,0,0],[1,1,2,1]]` → `3` (các số cách: `3, 3, 3`).

**Ràng buộc**

- `2 <= n <= 2 * 10^4`, `parent[0] = -1`, `0 <= parent[i] < n`, đầu vào là cây hợp lệ
- `0 <= red_i, blue_i, white_i <= 10`
- `1 <= queries.length <= 2 * 10^4`, `0 <= aNode, bNode <= n - 1`, `aCard, bCard` thuộc `{0, 1}`

Lưu ý: `parent[i]` có thể lớn hơn `i`, nên **không** được giả sử cha luôn có chỉ số nhỏ hơn con.

In [1]:
examples = [
    (dict(n=3, parent=[-1, 0, 0], gates=[[1, 0, 1], [0, 1, 1], [1, 1, 0]],
          queries=[[1, 0, 2, 0], [1, 1, 2, 0], [1, 0, 2, 1]]), 1),
    (dict(n=3, parent=[-1, 0, 1], gates=[[0, 1, 2], [1, 0, 1], [0, 0, 3]],
          queries=[[2, 0, 1, 0], [2, 1, 0, 0], [1, 1, 2, 1]]), 3),
]
for args, expected in examples:
    print("n =", args["n"], "| parent =", args["parent"], "| số truy vấn =", len(args["queries"]), "| mong đợi:", expected)

n = 3 | parent = [-1, 0, 0] | số truy vấn = 3 | mong đợi: 1
n = 3 | parent = [-1, 0, 1] | số truy vấn = 3 | mong đợi: 3


## 3. Kiến thức nền

### 3.1. Cây bằng mảng `parent`, danh sách con và độ sâu

Mảng `parent` cho biết cha của mỗi nút. Từ đó dựng `children[u]` (danh sách con) và tính `depth[u]` (độ sâu, gốc có độ sâu `0`) bằng BFS từ gốc. **Không** dựa vào thứ tự chỉ số, vì `parent[i]` có thể lớn hơn `i`.

In [2]:
parent = [-1, 0, 0, 1, 1, 2]       # 0 là gốc; 1,2 là con của 0; 3,4 con của 1; 5 con của 2
n = len(parent)
children = [[] for _ in range(n)]
for v in range(1, n):
    children[parent[v]].append(v)
print("children:", children)

depth = [0] * n
order = [0]                          # BFS: vừa duyệt vừa thêm vào cuối list
for u in order:
    for c in children[u]:
        depth[c] = depth[u] + 1
        order.append(c)
print("thứ tự BFS:", order)
print("độ sâu   :", depth)

children: [[1, 2], [3, 4], [5], [], [], []]
thứ tự BFS: [0, 1, 2, 3, 4, 5]
độ sâu   : [0, 1, 1, 2, 2, 2]


### 3.2. LCA (tổ tiên chung gần nhất) bằng cách leo từng bước

Cách đơn giản nhất: nâng nút sâu hơn lên cho cùng độ sâu, rồi cả hai cùng leo cho tới khi gặp nhau.

In [3]:
def lca_naive(a, b, parent, depth):
    while depth[a] > depth[b]:
        a = parent[a]
    while depth[b] > depth[a]:
        b = parent[b]
    while a != b:
        a, b = parent[a], parent[b]
    return a

print(lca_naive(3, 4, parent, depth))   # 1
print(lca_naive(3, 5, parent, depth))   # 0
print(lca_naive(3, 1, parent, depth))   # 1  (nút là tổ tiên của nút kia)

1
0
1


### 3.3. Đếm đường đi có trạng thái bằng ma trận `2×2`

Trạng thái của một người chỉ là màu thẻ: `0` (xanh) hoặc `1` (đỏ). Tại một nút có `red = r`, `blue = b`, `white = w`, số cách chuyển:

| từ \ sang | xanh (0) | đỏ (1) |
|---|---|---|
| xanh (0) | `b` (cổng xanh) | `w` (cổng trắng) |
| đỏ (1) | `w` (cổng trắng) | `r` (cổng đỏ) |

Viết thành ma trận `T = [[b, w], [w, r]]` (hàng = màu hiện tại, cột = màu sau). Nếu vectơ hàng `x = (x0, x1)` là "số cách tới từng màu thẻ", thì sau khi qua nút: `x' = x · T`. Qua **hai** nút liên tiếp: `x · T1 · T2`. Đó là lý do ta nhân ma trận theo thứ tự **từ nút xuất phát lên trên**.

In [4]:
MOD = 10**9 + 7

def gate_matrix(g):
    r, b, w = g
    # (m00, m01, m10, m11); chỉ số 0 = xanh, 1 = đỏ
    return (b, w, w, r)

def vec_mul(x, T):
    return ((x[0] * T[0] + x[1] * T[2]) % MOD, (x[0] * T[1] + x[1] * T[3]) % MOD)

def mat_mul(A, B):
    return ((A[0] * B[0] + A[1] * B[2]) % MOD, (A[0] * B[1] + A[1] * B[3]) % MOD,
            (A[2] * B[0] + A[3] * B[2]) % MOD, (A[2] * B[1] + A[3] * B[3]) % MOD)

T = gate_matrix([2, 3, 4])           # 2 đỏ, 3 xanh, 4 trắng
print("T =", T)                      # (3, 4, 4, 2)
x = (1, 0)                           # đang cầm thẻ xanh, 1 cách
print("sau 1 nút:", vec_mul(x, T))   # (3, 4): 3 cách giữ xanh, 4 cách đổi sang đỏ

T = (3, 4, 4, 2)
sau 1 nút: (3, 4)


### 3.4. Phép nhân ma trận cho dãy nút và tính kết hợp

Nhân ma trận **không giao hoán** (`A·B ≠ B·A`) nhưng **có tính kết hợp**: `(A·B)·C = A·(B·C)`. Nhờ kết hợp, ta có thể gộp trước các đoạn nút liền nhau thành một ma trận (đúng như binary lifting sẽ làm ở phần 7).

In [5]:
A = gate_matrix([1, 0, 1]); B = gate_matrix([0, 1, 1]); C = gate_matrix([1, 1, 0])
print("kết hợp   :", mat_mul(mat_mul(A, B), C) == mat_mul(A, mat_mul(B, C)))
print("giao hoán?:", mat_mul(A, B) == mat_mul(B, A))

kết hợp   : True
giao hoán?: False


### 3.5. Phân rã số bước thành các lũy thừa của 2 (binary lifting)

Muốn leo `d` bước, tách `d` thành các bit: `d = 13 = 8 + 4 + 1`. Nếu đã có sẵn "bước nhảy 1, 2, 4, 8, ..." thì chỉ cần nhảy theo các bit bằng `1` của `d`. Ta cũng dùng cách này để duyệt bit.

In [6]:
d = 13
k = 0
while d >> k:
    if d >> k & 1:
        print(f"bit {k} bật -> nhảy {1 << k} bước")
    k += 1
print((13).bit_length(), "bit")

bit 0 bật -> nhảy 1 bước
bit 2 bật -> nhảy 4 bước
bit 3 bật -> nhảy 8 bước
4 bit


## 4. Ý tưởng nền tảng / trực giác

**Tính tay ví dụ 1, truy vấn 0:** `Alice = (nút 1, xanh)`, `Bob = (nút 2, xanh)`, `LCA = 0`.

- Alice rời nút `1` (`gates[1] = [0,1,1]`: 0 đỏ, 1 xanh, 1 trắng). Thẻ xanh: dùng cổng xanh (1 cách), hoặc cổng trắng (1 cách) → **2 cách**.
- Bob rời nút `2` (`gates[2] = [1,1,0]`): thẻ xanh: cổng xanh (1 cách); không có trắng → **1 cách**.
- Tích: `2 × 1 = 2`. ✓

**Tính tay ví dụ 2, truy vấn 1:** `Alice = (nút 2, đỏ)`, `Bob = (nút 0, xanh)`, `LCA = 0`; đường đi của Alice `2 → 1 → 0`.

- Rời nút `2` (`[0,0,3]`) với thẻ đỏ: không có cổng đỏ; 3 cổng trắng đổi sang xanh → `3` cách, thẻ thành xanh.
- Rời nút `1` (`[1,0,1]`) với thẻ xanh: không có cổng xanh; 1 cổng trắng đổi sang đỏ → `1` cách.
- Alice: `3 × 1 = 3`; Bob đã ở LCA: `1` → tổng `3`. ✓

Từ đó rút ra: số cách của một người = **tích số cách chọn cổng tại từng nút trên đường đi**, nhưng "số cách chọn cổng ở nút sau" phụ thuộc màu thẻ do lựa chọn trước tạo ra ⇒ cần cộng theo màu thẻ ⇒ nhân ma trận.

## 5. Cách 1 — Vét cạn: liệt kê từng cổng cụ thể

**Ý tưởng.** Đúng nghĩa đen của đề: đệ quy `go(u, màu)`; nếu `u == l` trả `1`; ngược lại duyệt **từng cổng** của `u` (mỗi cổng là một nhánh riêng) và đệ quy lên `parent[u]` với màu mới. Cuối cùng nhân kết quả của Alice và Bob, XOR các truy vấn.

Cách này chỉ dùng cho cây rất nhỏ (số cổng nhỏ, đường đi ngắn) vì số nhánh nhân lên theo từng nút.

In [7]:
def ways_enum(v, card, l, parent, gates):
    """Đếm bằng cách duyệt từng cổng cụ thể (mỗi cổng một nhánh)."""
    def go(u, c):
        if u == l:
            return 1
        r, b, w = gates[u]
        total = 0
        for kind in ["R"] * r + ["B"] * b + ["W"] * w:      # từng cổng, kể cả cổng giống nhau
            if kind == "R" and c == 1:
                total += go(parent[u], 1)
            elif kind == "B" and c == 0:
                total += go(parent[u], 0)
            elif kind == "W":
                total += go(parent[u], 1 - c)
        return total
    return go(v, card)

def brute_force(n, parent, gates, queries):
    depth = [0] * n
    children = [[] for _ in range(n)]
    for v in range(1, n):
        children[parent[v]].append(v)
    order = [0]
    for u in order:
        for c in children[u]:
            depth[c] = depth[u] + 1
            order.append(c)
    res = 0
    per_query = []
    for a, ca, b, cb in queries:
        l = lca_naive(a, b, parent, depth)
        w = ways_enum(a, ca, l, parent, gates) * ways_enum(b, cb, l, parent, gates) % MOD
        per_query.append(w)
        res ^= w
    return res, per_query

for args, expected in examples:
    got, per_q = brute_force(**args)
    print("từng truy vấn:", per_q, "-> XOR =", got, "(mong đợi", expected, ")")
    assert got == expected

từng truy vấn: [2, 1, 2] -> XOR = 1 (mong đợi 1 )
từng truy vấn: [3, 3, 3] -> XOR = 3 (mong đợi 3 )


**Độ phức tạp.** Số nhánh là tích `(r+b+w)` qua các nút trên đường đi ⇒ mũ theo độ dài đường đi. Không dùng được cho `n = 2·10^4`.

**Nút thắt.** Rất nhiều nhánh dẫn tới cùng một trạng thái "đang ở nút `u` với thẻ màu `c`" — chỉ khác ở **số lượng** nhánh. Ta không cần liệt kê từng nhánh, chỉ cần **đếm** số nhánh ứng với mỗi màu thẻ.

## 6. Bước đột phá

**Insight 1 — gộp theo màu thẻ.** Gọi `ways[c]` là số cách dẫn tới nút hiện tại với thẻ màu `c`. Qua một nút với `(r, b, w)`:

- `ways'[xanh] = ways[xanh] · b + ways[đỏ] · w`
- `ways'[đỏ]  = ways[xanh] · w + ways[đỏ] · r`

Đây là `x' = x · T` với `T = [[b, w], [w, r]]`.

**Insight 2 — số cách của một người = tổng các phần tử vectơ cuối.** Sau khi qua mọi nút trên đường tới LCA, ta không quan tâm màu thẻ nữa nên cộng `x0 + x1`.

**Insight 3 — một truy vấn là nhân một dãy ma trận dọc đường đi lên.** Đường đi `a → ... → (con của l)` gồm `d = depth[a] - depth[l]` nút. Nếu mỗi truy vấn tự leo từng nút thì tốn `O(d)`; với `2·10^4` truy vấn trên cây là một đường thẳng dài `2·10^4` nút thì tới `4·10^8` phép nhân ma trận — quá chậm.

**Insight 4 — nhân đôi (binary lifting) cho tích ma trận.** Với mỗi nút `v` và mỗi `k`, lưu sẵn:

- `anc[k][v]` = tổ tiên cách `v` đúng `2^k` bước;
- `mat[k][v]` = tích ma trận của `2^k` nút liên tiếp `v, parent(v), ...` (đúng thứ tự leo).

Khi đó: `mat[k][v] = mat[k-1][v] · mat[k-1][anc[k-1][v]]` và leo `d` bước chỉ cần `O(log d)` lần nhân **vectơ × ma trận** (rẻ hơn nhân ma trận).

**Tại sao không dùng "tiền tố + ma trận nghịch đảo" như tổng tiền tố số?** Vì `T = [[b, w], [w, r]]` có định thức `b·r − w²` có thể `= 0 (mod 10^9+7)` — không khả nghịch. Ví dụ nút có `r = b = w = 1` có `det = 0`. Vì thế ta phải nhân **đúng đoạn** chứ không "chia" được.

**Quy tắc "nếu… thì…":**

- Nếu `a == l` (Alice đã ở LCA) thì `d = 0`, vectơ giữ nguyên, số cách `= 1`.
- Nếu một nút trên đường đi không có cổng dùng được cho màu hiện tại thì phần tử tương ứng của ma trận bằng `0`, tích tự động ra `0`.
- Nhảy bước lớn **không** bao giờ vượt quá `d` nên không đụng tới gốc (gốc không có cổng "ra").

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đi bộ từng nút, nhân vectơ với ma trận (`O(độ sâu)` mỗi truy vấn)

**Giải thích từng đoạn.**

1. Dựng `depth` bằng BFS.
2. Với mỗi truy vấn: `l = LCA(a, b)` (leo từng bước).
3. Mỗi người bắt đầu từ vectơ one-hot theo màu thẻ: xanh `(1, 0)`, đỏ `(0, 1)`.
4. Leo `u = a`; trong khi `u != l`: `x = x · gate_matrix(gates[u])`, `u = parent[u]`.
5. Số cách = `(x0 + x1) mod p`.

Cách này **đúng** và là cơ sở để kiểm tra các cách nhanh hơn trên cây vừa.

In [8]:
def ways_walk(v, card, l, parent, gates):
    x = (1, 0) if card == 0 else (0, 1)
    u = v
    while u != l:
        x = vec_mul(x, gate_matrix(gates[u]))
        u = parent[u]
    return (x[0] + x[1]) % MOD

def dp_walk(n, parent, gates, queries):
    children = [[] for _ in range(n)]
    for v in range(1, n):
        children[parent[v]].append(v)
    depth = [0] * n
    order = [0]
    for u in order:
        for c in children[u]:
            depth[c] = depth[u] + 1
            order.append(c)
    res = 0
    for a, ca, b, cb in queries:
        l = lca_naive(a, b, parent, depth)
        res ^= ways_walk(a, ca, l, parent, gates) * ways_walk(b, cb, l, parent, gates) % MOD
    return res

for args, expected in examples:
    got = dp_walk(**args)
    print(got, expected)
    assert got == expected

1 1
3 3


**Độ phức tạp.** Mỗi truy vấn tốn `O(depth)` ⇒ tổng `O(q · n)` trường hợp xấu (cây là đường thẳng). `2·10^4 × 2·10^4 = 4·10^8` — quá chậm.

### 7.2. Cách 3 — Binary lifting (bản chính, `O((n + q) log n)`)

**Giải thích từng đoạn.**

1. **Dựng cây:** `children`, `depth`, thứ tự BFS từ gốc.
2. **Bảng nhảy:** `LOG = max(1, độ sâu lớn nhất).bit_length()`.
   - Tầng 0: `anc[0][v] = parent[v]` (gốc trỏ tới chính nó cho an toàn); `mat[0][v] = gate_matrix(gates[v])`.
   - Tầng `k`: `anc[k][v] = anc[k-1][ anc[k-1][v] ]`; `mat[k][v] = mat[k-1][v] · mat[k-1][ anc[k-1][v] ]`.
   - Với nút mà `2^k` vượt quá độ sâu của nó, giá trị là rác nhưng **không bao giờ được dùng** vì ta chỉ nhảy `d <= depth[v]` bước.
3. **LCA:** nâng nút sâu hơn lên cho cùng độ sâu bằng các bước nhảy; nếu trùng thì xong; ngược lại cho cả hai nhảy từ bit cao xuống thấp khi `anc` khác nhau.
4. **Leo `d` bước với vectơ:** với mỗi bit `k` của `d`: `x = x · mat[k][cur]`, `cur = anc[k][cur]`.
5. **Đáp án truy vấn:** `(Alice) · (Bob) mod p`, rồi XOR dồn.

In [9]:
def build_lifting(n, parent, gates):
    children = [[] for _ in range(n)]
    for v in range(1, n):
        children[parent[v]].append(v)
    depth = [0] * n
    order = [0]
    for u in order:
        for c in children[u]:
            depth[c] = depth[u] + 1
            order.append(c)
    LOG = max(1, max(depth).bit_length())
    anc = [[0] * n for _ in range(LOG)]
    mat = [[None] * n for _ in range(LOG)]
    for v in range(n):
        anc[0][v] = parent[v] if v else 0              # gốc trỏ về chính nó
        mat[0][v] = gate_matrix(gates[v])
    for k in range(1, LOG):
        a1, m1, a0, m0 = anc[k], mat[k], anc[k - 1], mat[k - 1]
        for v in range(n):
            mid = a0[v]
            a1[v] = a0[mid]
            m1[v] = mat_mul(m0[v], m0[mid])
    return depth, anc, mat, LOG

def lca_lifting(a, b, depth, anc, LOG):
    if depth[a] < depth[b]:
        a, b = b, a
    diff = depth[a] - depth[b]
    for k in range(LOG):
        if diff >> k & 1:
            a = anc[k][a]
    if a == b:
        return a
    for k in range(LOG - 1, -1, -1):
        if anc[k][a] != anc[k][b]:
            a, b = anc[k][a], anc[k][b]
    return anc[0][a]

def climb(v, card, d, anc, mat, LOG):
    """Số cách leo d bước từ v với thẻ card."""
    x = (1, 0) if card == 0 else (0, 1)
    k = 0
    while d >> k:
        if d >> k & 1:
            x = vec_mul(x, mat[k][v])
            v = anc[k][v]
        k += 1
    return (x[0] + x[1]) % MOD

def dp_lifting(n, parent, gates, queries):
    depth, anc, mat, LOG = build_lifting(n, parent, gates)
    res = 0
    for a, ca, b, cb in queries:
        l = lca_lifting(a, b, depth, anc, LOG)
        wa = climb(a, ca, depth[a] - depth[l], anc, mat, LOG)
        wb = climb(b, cb, depth[b] - depth[l], anc, mat, LOG)
        res ^= wa * wb % MOD
    return res

for args, expected in examples:
    got = dp_lifting(**args)
    print(got, expected)
    assert got == expected

1 1
3 3


**Độ phức tạp.** Dựng bảng: `O(n log n)` phép nhân ma trận `2×2`. Mỗi truy vấn: LCA `O(log n)` + hai lần leo `O(log n)`. Tổng `O((n + q) log n)`; bộ nhớ `O(n log n)` (mỗi ô là một tuple 4 số).

### 7.3. Cách 4 — Offline: DFS giữ "đường từ gốc" và segment tree theo độ sâu

**Ý tưởng khác.** Thay vì lưu bảng nhảy cho mọi nút, ta duyệt cây bằng DFS và luôn giữ **đường từ gốc tới nút hiện tại** trong một segment tree theo chỉ số độ sâu: vị trí `d` chứa ma trận của nút ở độ sâu `d` trên đường hiện tại. Khi thăm nút `u`, các truy vấn "một nửa" bắt đầu từ `u` cần tích ma trận trên các độ sâu `depth[u], depth[u]-1, ..., depth[l]+1` (theo thứ tự **độ sâu giảm dần**) — đó là một truy vấn đoạn trên segment tree.

**Chi tiết.**

- Mỗi nút của segment tree lưu tích **theo thứ tự độ sâu giảm dần**: `val(node) = val(phải) · val(trái)` (phải = độ sâu lớn hơn).
- Duyệt DFS tiền thứ tự bằng stack: khi lấy `u` ra, ghi `mat(u)` vào vị trí `depth[u]`. Vì cây con của một tổ tiên liền nhau trong thứ tự tiền thứ tự nên các vị trí độ sâu nhỏ hơn `depth[u]` luôn đang giữ đúng các tổ tiên của `u`.
- Mỗi truy vấn tách thành hai nửa (Alice, Bob), gắn vào nút xuất phát; LCA tính trước bằng bất kỳ cách nào.

In [10]:
IDENT = (1, 0, 0, 1)

class SegTree:
    """Segment tree lưu tích ma trận theo thứ tự độ sâu GIẢM dần (phải · trái)."""
    def __init__(self, size):
        self.size = size
        self.t = [IDENT] * (4 * size)
    def set(self, pos, val, node=1, lo=0, hi=None):
        if hi is None:
            hi = self.size - 1
        if lo == hi:
            self.t[node] = val
            return
        mid = (lo + hi) // 2
        if pos <= mid:
            self.set(pos, val, 2 * node, lo, mid)
        else:
            self.set(pos, val, 2 * node + 1, mid + 1, hi)
        self.t[node] = mat_mul(self.t[2 * node + 1], self.t[2 * node])
    def query(self, ql, qr, node=1, lo=0, hi=None):
        if hi is None:
            hi = self.size - 1
        if ql > hi or qr < lo:
            return IDENT
        if ql <= lo and hi <= qr:
            return self.t[node]
        mid = (lo + hi) // 2
        right = self.query(ql, qr, 2 * node + 1, mid + 1, hi)
        left = self.query(ql, qr, 2 * node, lo, mid)
        return mat_mul(right, left)

def dp_offline(n, parent, gates, queries):
    depth, anc, mat, LOG = build_lifting(n, parent, gates)       # chỉ dùng để tính depth và LCA
    children = [[] for _ in range(n)]
    for v in range(1, n):
        children[parent[v]].append(v)
    half = [[] for _ in range(n)]                                # half[u] = [(qi, người, l, thẻ)]
    for qi, (a, ca, b, cb) in enumerate(queries):
        l = lca_lifting(a, b, depth, anc, LOG)
        half[a].append((qi, 0, l, ca))
        half[b].append((qi, 1, l, cb))
    seg = SegTree(max(depth) + 1)
    ways = [[1, 1] for _ in queries]
    stack = [0]
    while stack:
        u = stack.pop()
        seg.set(depth[u], gate_matrix(gates[u]))
        for qi, who, l, card in half[u]:
            if depth[u] > depth[l]:
                P = seg.query(depth[l] + 1, depth[u])            # tích theo độ sâu giảm dần
                x = (1, 0) if card == 0 else (0, 1)
                x = vec_mul(x, P)
                ways[qi][who] = (x[0] + x[1]) % MOD
        stack.extend(children[u])
    res = 0
    for wa, wb in ways:
        res ^= wa * wb % MOD
    return res

for args, expected in examples:
    got = dp_offline(**args)
    print(got, expected)
    assert got == expected

1 1
3 3


**Độ phức tạp.** Mỗi nút một lần `set` (`O(log n)`), mỗi nửa truy vấn một lần `query` (`O(log n)` phép nhân ma trận). Tổng `O((n + q) log n)`. Cách này hữu ích khi ta không muốn tốn `O(n log n)` bộ nhớ cho bảng nhảy, nhưng code dài hơn và cần biết trước toàn bộ truy vấn (offline).

## 8. TRACE từng bước

Ta trace cách 2 cho ví dụ 2, truy vấn 1: `Alice = (nút 2, đỏ)`, LCA `= 0`. In vectơ `x = (xanh, đỏ)` sau từng nút.

Bảng mong đợi (`gates[2] = [0,0,3]`, `gates[1] = [1,0,1]`):

| Bước | Nút đang rời | Ma trận `T = (b, w, w, r)` | Vectơ trước | Vectơ sau |
|---|---|---|---|---|
| 0 | (bắt đầu) | — | — | `(0, 1)` (đỏ) |
| 1 | `2` | `(0, 3, 3, 0)` | `(0, 1)` | `(3, 0)` |
| 2 | `1` | `(0, 1, 1, 1)` | `(3, 0)` | `(0, 3)` |

Tổng `0 + 3 = 3` ✓.

In [11]:
def trace_walk(v, card, l, parent, gates):
    x = (1, 0) if card == 0 else (0, 1)
    print(f"bắt đầu tại nút {v}, thẻ {'đỏ' if card else 'xanh'}, đích LCA = {l}; vectơ (xanh, đỏ) = {x}")
    u, step = v, 0
    while u != l:
        step += 1
        r, b, w = gates[u]
        T = gate_matrix(gates[u])
        nx = vec_mul(x, T)
        print(f"  bước {step}: rời nút {u} (đỏ={r}, xanh={b}, trắng={w}), T={T}: {x} -> {nx}")
        x, u = nx, parent[u]
    ans = (x[0] + x[1]) % MOD
    print("  tổng các phần tử =", ans)
    return ans

args = examples[1][0]
trace_walk(2, 1, 0, args["parent"], args["gates"])

bắt đầu tại nút 2, thẻ đỏ, đích LCA = 0; vectơ (xanh, đỏ) = (0, 1)
  bước 1: rời nút 2 (đỏ=0, xanh=0, trắng=3), T=(0, 3, 3, 0): (0, 1) -> (3, 0)
  bước 2: rời nút 1 (đỏ=1, xanh=0, trắng=1), T=(0, 1, 1, 1): (3, 0) -> (0, 3)
  tổng các phần tử = 3


3

In [12]:
# Trace binary lifting: cây là một đường thẳng 0 <- 1 <- 2 <- ... <- 6, leo từ nút 6 lên gốc 0 (d = 6 = 4 + 2)
parent_line = [-1, 0, 1, 2, 3, 4, 5]
gates_line = [[1, 1, 1]] * 7
depth, anc, mat, LOG = build_lifting(7, parent_line, gates_line)
print("độ sâu:", depth, "| LOG =", LOG)
print("anc[0] =", anc[0])
print("anc[1] =", anc[1], "  (nhảy 2 bước)")
print("anc[2] =", anc[2], "  (nhảy 4 bước; nút nông hơn 4 là 'rác' không dùng)")
d = depth[6] - depth[0]
x, cur = (0, 1), 6
for k in range(LOG):
    if d >> k & 1:
        print(f"bit {k}: nhảy {1<<k} bước từ nút {cur} -> {anc[k][cur]}, dùng mat[{k}][{cur}] = {mat[k][cur]}")
        x = vec_mul(x, mat[k][cur]); cur = anc[k][cur]
print("kết quả:", x, "tổng =", sum(x) % MOD, "| đi bộ từng nút:", ways_walk(6, 1, 0, parent_line, gates_line))

độ sâu: [0, 1, 2, 3, 4, 5, 6] | LOG = 3
anc[0] = [0, 0, 1, 2, 3, 4, 5]
anc[1] = [0, 0, 0, 1, 2, 3, 4]   (nhảy 2 bước)
anc[2] = [0, 0, 0, 0, 0, 1, 2]   (nhảy 4 bước; nút nông hơn 4 là 'rác' không dùng)
bit 1: nhảy 2 bước từ nút 6 -> 4, dùng mat[1][6] = (2, 2, 2, 2)
bit 2: nhảy 4 bước từ nút 4 -> 0, dùng mat[2][4] = (8, 8, 8, 8)
kết quả: (32, 32) tổng = 64 | đi bộ từng nút: 64


Với cây đường thẳng và mọi nút có `(1,1,1)` (ma trận `[[1,1],[1,1]]`), mỗi nút nhân đôi số cách tổng (`x0 + x1` nhân 2 qua mỗi nút): 6 nút ⇒ `2^6 = 64` cách. Binary lifting chỉ cần 2 lần nhảy (bit 1 và bit 2) thay vì 6 lần nhân.

## 9. Code nộp LeetCode

Dùng binary lifting (cách 3), gọn trong một phương thức. Ghi chú: do là bài Premium, tên phương thức `distinctGatePaths` là suy luận — hãy đối chiếu lại nếu có quyền xem.

In [13]:
class Solution:
    def distinctGatePaths(self, n: int, parent: list[int], gates: list[list[int]], queries: list[list[int]]) -> int:
        MOD = 10**9 + 7
        children = [[] for _ in range(n)]
        for v in range(1, n):
            children[parent[v]].append(v)
        depth = [0] * n
        order = [0]
        for u in order:                              # BFS từ gốc, không dựa vào thứ tự chỉ số
            for c in children[u]:
                depth[c] = depth[u] + 1
                order.append(c)
        LOG = max(1, max(depth).bit_length())
        anc = [[0] * n for _ in range(LOG)]
        mat = [[None] * n for _ in range(LOG)]
        for v in range(n):
            anc[0][v] = parent[v] if v else 0
            r, b, w = gates[v]
            mat[0][v] = (b, w, w, r)                  # (xanh->xanh, xanh->đỏ, đỏ->xanh, đỏ->đỏ)
        for k in range(1, LOG):
            for v in range(n):
                mid = anc[k - 1][v]
                anc[k][v] = anc[k - 1][mid]
                A, B = mat[k - 1][v], mat[k - 1][mid]
                mat[k][v] = ((A[0] * B[0] + A[1] * B[2]) % MOD, (A[0] * B[1] + A[1] * B[3]) % MOD,
                             (A[2] * B[0] + A[3] * B[2]) % MOD, (A[2] * B[1] + A[3] * B[3]) % MOD)

        def climb(v, card, d):
            x0, x1 = (1, 0) if card == 0 else (0, 1)
            k = 0
            while d >> k:
                if d >> k & 1:
                    T = mat[k][v]
                    x0, x1 = (x0 * T[0] + x1 * T[2]) % MOD, (x0 * T[1] + x1 * T[3]) % MOD
                    v = anc[k][v]
                k += 1
            return (x0 + x1) % MOD

        ans = 0
        for a, ca, b, cb in queries:
            x, y = a, b
            if depth[x] < depth[y]:
                x, y = y, x
            diff = depth[x] - depth[y]
            for k in range(LOG):
                if diff >> k & 1:
                    x = anc[k][x]
            if x != y:
                for k in range(LOG - 1, -1, -1):
                    if anc[k][x] != anc[k][y]:
                        x, y = anc[k][x], anc[k][y]
                x = anc[0][x]
            l = x                                     # LCA
            ans ^= climb(a, ca, depth[a] - depth[l]) * climb(b, cb, depth[b] - depth[l]) % MOD
        return ans


sol = Solution()
for args, expected in examples:
    got = sol.distinctGatePaths(**args)
    print(got, expected)
    assert got == expected

1 1
3 3


## 10. Vì sao thuật toán đúng

**Đếm đúng một người.** Gọi `x_t[c]` là số danh sách lựa chọn cổng (mỗi cổng cụ thể là một lựa chọn) cho `t` nút đầu của đường đi sao cho thẻ cuối cùng có màu `c`. Một danh sách cho `t+1` nút được tạo duy nhất từ một danh sách cho `t` nút cộng thêm một cổng ở nút thứ `t+1`. Số cổng khả dụng từ màu `c` sang màu `c'` là phần tử `T[c][c']` (cổng cùng màu cho `c' = c`, cổng trắng cho `c' ≠ c`). Do đó `x_{t+1}[c'] = Σ_c x_t[c] · T[c][c']`, tức `x_{t+1} = x_t · T`. Tổng `x[0] + x[1]` cuối cùng đếm đủ mọi danh sách hợp lệ, **mỗi danh sách đúng một lần** (hai danh sách khác nhau ở một cổng thì khác nhau, kể cả cổng cùng loại).

**Hai người độc lập.** Mỗi người leo trên đoạn đường **không giao nhau** tính theo nút rời (Alice rời các nút từ `a` tới con của `l` trên nhánh của mình, Bob tương tự; LCA không phải nút "rời" của ai). Tập cổng sử dụng của Alice và của Bob độc lập, nên tổng số cách = tích.

**Binary lifting đúng.** `mat[k][v]` = tích ma trận của `2^k` nút `v, parent(v), ...` theo đúng thứ tự leo (quy nạp bằng tính kết hợp của phép nhân ma trận). Với `d <= depth[v]`, phân rã `d` thành bit rồi nhảy từ bit thấp lên cao tích đúng `d` nút theo thứ tự leo: các đoạn `2^k` nối tiếp nhau. Phần "rác" ở các nút nông không bao giờ bị đụng vì chỉ nhảy khi `d` có bit đó, và tổng các bước đã nhảy luôn `<= d <= depth[v]`.

**LCA đúng.** Cách nâng nút sâu hơn rồi nhảy từ bit cao xuống thấp khi hai tổ tiên khác nhau là phương pháp chuẩn: nó tìm nút sâu nhất mà hai nhánh còn chưa gặp nhau, cha của nó là LCA.

**Cách 4 (offline).** DFS tiền thứ tự bằng stack duy trì bất biến: khi xử lý `u`, các vị trí độ sâu `1..depth[u]` chứa đúng đường từ gốc tới `u`. Truy vấn đoạn `[depth[l]+1, depth[u]]` theo thứ tự độ sâu giảm dần chính là tích ma trận theo thứ tự leo.

## 11. Phân tích Time/Space

Đặt `n` = số nút, `q` = số truy vấn, `D` = độ sâu lớn nhất.

| Cách | Time | Space |
|---|---|---|
| Liệt kê cổng | mũ theo độ dài đường đi | `O(D)` |
| Đi bộ từng nút | `O(n + q · D)` | `O(n)` |
| Binary lifting (nộp) | `O((n + q) log n)` | `O(n log n)` |
| Offline + segment tree | `O((n + q) log n)` | `O(n)` |

Với `n, q = 2·10^4` và `log n = 15`: khoảng `3·10^5` phép nhân ma trận để dựng bảng + `~10^6` phép nhân vectơ cho các truy vấn.

In [14]:
import time, random
rng = random.Random(3)
n = 20000
# cây là một đường thẳng sâu (xấu nhất cho cách đi bộ), cha có chỉ số lớn hơn con ở một số nơi để thử tính tổng quát
perm = list(range(1, n))
rng.shuffle(perm)
labels = [0] + perm                      # nhãn của nút ở độ sâu d là labels[d]
parent = [-1] * n
for d in range(1, n):
    parent[labels[d]] = labels[d - 1]
gates = [[rng.randint(0, 10) for _ in range(3)] for _ in range(n)]
queries = [[rng.randrange(n), rng.randint(0, 1), rng.randrange(n), rng.randint(0, 1)] for _ in range(20000)]
t0 = time.time()
res = Solution().distinctGatePaths(n, parent, gates, queries)
print("Solution (binary lifting):", res, f"- {time.time()-t0:.2f}s")
t0 = time.time()
res2 = dp_offline(n, parent, gates, queries)
print("offline + segment tree   :", res2, f"- {time.time()-t0:.2f}s")
assert res == res2

Solution (binary lifting): 510150457 - 0.39s


offline + segment tree   : 510150457 - 0.82s


## 12. Bộ test

Chiến lược test:

1. 2 ví dụ của đề (assert), kèm đáp án **từng truy vấn** `2,1,2` và `3,3,3` ở ví dụ.
2. Biên: `a == b` (LCA là chính nó); `a` là tổ tiên của `b`; nút không có cổng dùng được (kết quả `0`); toàn cổng trắng.
3. Phản ví dụ cho lỗi hay gặp (phần 13).
4. Đối chiếu: liệt kê cổng, đi bộ, binary lifting, offline và `Solution` trên cây ngẫu nhiên (seed cố định), trong đó `parent[i]` **không** nhất thiết nhỏ hơn `i`.

In [15]:
def per_query_values(n, parent, gates, queries):
    return brute_force(n, parent, gates, queries)[1]

assert per_query_values(**examples[0][0]) == [2, 1, 2]
assert per_query_values(**examples[1][0]) == [3, 3, 3]

sol = Solution()
for args, expected in examples:
    assert sol.distinctGatePaths(**args) == expected

# a == b -> LCA = chính nó -> cả hai có đúng 1 cách
assert sol.distinctGatePaths(2, [-1, 0], [[0, 0, 0], [0, 0, 0]], [[1, 0, 1, 1]]) == 1
# a là tổ tiên của b: Alice 1 cách; Bob từ nút 2 lên nút 1 với thẻ xanh, gates[2] = [0,2,3] -> 2 xanh + 3 trắng = 5
assert sol.distinctGatePaths(3, [-1, 0, 1], [[0, 0, 0], [0, 0, 0], [0, 2, 3]], [[1, 1, 2, 0]]) == 5
# không có cổng dùng được: nút 1 chỉ có cổng đỏ, Alice cầm thẻ xanh -> 0 cách
assert sol.distinctGatePaths(2, [-1, 0], [[5, 5, 5], [4, 0, 0]], [[1, 0, 0, 0]]) == 0
# toàn cổng trắng: mỗi nút w cách, đường dài 3 nút -> w1*w2*w3 bất kể màu
assert sol.distinctGatePaths(4, [-1, 0, 1, 2], [[0, 0, 0], [0, 0, 2], [0, 0, 3], [0, 0, 5]], [[3, 1, 0, 0]]) == 30
# parent[i] > i vẫn đúng: cây 0 <- 2 <- 1 (cha của 1 là 2), gates[1]=[0,3,0], gates[2]=[0,0,4]
assert sol.distinctGatePaths(3, [-1, 2, 0], [[0, 0, 0], [0, 3, 0], [0, 0, 4]], [[1, 0, 0, 0]]) == 12
print("test cố định OK")

test cố định OK


In [16]:
def random_tree(rng, n):
    """parent[i] không nhất thiết < i: tạo cây rồi đánh nhãn lại ngẫu nhiên, gốc vẫn là 0."""
    labels = list(range(1, n))
    rng.shuffle(labels)
    labels = [0] + labels
    parent = [-1] * n
    for i in range(1, n):
        parent[labels[i]] = labels[rng.randrange(i)]
    return parent

rng = random.Random(8888)
cnt = 0
for _ in range(1200):
    n = rng.randint(2, 7)
    parent = random_tree(rng, n)
    gates = [[rng.randint(0, 3) for _ in range(3)] for _ in range(n)]
    queries = [[rng.randrange(n), rng.randint(0, 1), rng.randrange(n), rng.randint(0, 1)] for _ in range(rng.randint(1, 5))]
    args = dict(n=n, parent=parent, gates=gates, queries=queries)
    expected = brute_force(**args)[0]
    assert dp_walk(**args) == expected, args
    assert dp_lifting(**args) == expected, args
    assert dp_offline(**args) == expected, args
    assert Solution().distinctGatePaths(**args) == expected, args
    cnt += 1
print("đã đối chiếu", cnt, "cây ngẫu nhiên nhỏ: tất cả khớp liệt kê cổng")

đã đối chiếu 1200 cây ngẫu nhiên nhỏ: tất cả khớp liệt kê cổng


In [17]:
# Cây vừa/lớn hơn: so cách đi bộ với binary lifting, offline và Solution (cổng 0..10, cây sâu)
rng = random.Random(4242)
for _ in range(25):
    n = rng.randint(30, 400)
    parent = random_tree(rng, n)
    gates = [[rng.randint(0, 10) for _ in range(3)] for _ in range(n)]
    queries = [[rng.randrange(n), rng.randint(0, 1), rng.randrange(n), rng.randint(0, 1)] for _ in range(60)]
    args = dict(n=n, parent=parent, gates=gates, queries=queries)
    e = dp_walk(**args)
    assert dp_lifting(**args) == e == dp_offline(**args) == Solution().distinctGatePaths(**args)
print("đi bộ == binary lifting == offline == Solution trên 25 cây vừa")

đi bộ == binary lifting == offline == Solution trên 25 cây vừa


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Coi các cổng cùng loại là một (không đếm riêng)

**Code sai:** nhân `1` thay vì nhân số lượng cổng, hoặc dùng `min(r, 1)`.
**Vì sao sai:** đề nói "identical gates are treated as separate and counted individually". Ví dụ 1 có `2` cách vì có 1 cổng xanh + 1 cổng trắng; với 3 cổng trắng thì có `3` cách.
**Code đúng:** ma trận dùng đúng số lượng `r, b, w`.

In [18]:
def wrong_count_types(n, parent, gates, queries):
    # SAI: coi một loại có mặt là 1 cách
    g2 = [[min(x, 1) for x in g] for g in gates]
    return dp_walk(n, parent, g2, queries)

print("sai :", wrong_count_types(**examples[1][0]), " đúng:", dp_walk(**examples[1][0]))

sai : 1  đúng: 3


### Lỗi 2 — Cộng thay vì nhân qua các nút (hoặc quên màu thẻ thay đổi)

**Code sai:** số cách của một người = `Σ (r + b + w)` hoặc tích `Π (r + b + w)` bỏ qua màu thẻ.
**Vì sao sai:** số cổng dùng được ở nút kế tiếp phụ thuộc màu thẻ do cổng trước tạo ra. Cổng đỏ dùng được với thẻ đỏ, không dùng được với thẻ xanh.
**Code đúng:** theo dõi vectơ `(xanh, đỏ)` và nhân với ma trận.

In [19]:
def wrong_ignore_card(n, parent, gates, queries):
    depth = [0] * n
    children = [[] for _ in range(n)]
    for v in range(1, n):
        children[parent[v]].append(v)
    order = [0]
    for u in order:
        for c in children[u]:
            depth[c] = depth[u] + 1
            order.append(c)
    res = 0
    for a, ca, b, cb in queries:
        l = lca_naive(a, b, parent, depth)
        def ways(v):
            tot, u = 1, v
            while u != l:
                tot = tot * sum(gates[u]) % MOD          # SAI: bỏ qua màu thẻ
                u = parent[u]
            return tot
        res ^= ways(a) * ways(b) % MOD
    return res

print("sai :", wrong_ignore_card(**examples[0][0]), " đúng:", dp_walk(**examples[0][0]))

sai : 4  đúng: 1


### Lỗi 3 — Nhân ma trận sai thứ tự

**Code sai:** `x = T_cha · T_con` (hoặc dùng `mat[k-1][mid] · mat[k-1][v]`).
**Vì sao sai:** nhân ma trận không giao hoán; tích phải theo thứ tự **từ nút xuất phát đi lên**.
**Code đúng:** `mat[k][v] = mat[k-1][v] · mat[k-1][anc[k-1][v]]` với vectơ hàng nhân từ trái.

In [20]:
def wrong_order_lifting(n, parent, gates, queries):
    depth, anc, mat, LOG = build_lifting(n, parent, gates)
    for k in range(1, LOG):                       # SAI: ghi đè bảng với thứ tự nhân ngược
        for v in range(n):
            mid = anc[k - 1][v]
            mat[k][v] = mat_mul(mat[k - 1][mid], mat[k - 1][v])
    res = 0
    for a, ca, b, cb in queries:
        l = lca_lifting(a, b, depth, anc, LOG)
        res ^= climb(a, ca, depth[a] - depth[l], anc, mat, LOG) * climb(b, cb, depth[b] - depth[l], anc, mat, LOG) % MOD
    return res

rng = random.Random(1)
for _ in range(500):
    n = rng.randint(4, 9)
    parent = random_tree(rng, n)
    gates = [[rng.randint(0, 3) for _ in range(3)] for _ in range(n)]
    queries = [[rng.randrange(n), rng.randint(0, 1), rng.randrange(n), rng.randint(0, 1)] for _ in range(4)]
    args = dict(n=n, parent=parent, gates=gates, queries=queries)
    if wrong_order_lifting(**args) != dp_walk(**args):
        print("phản ví dụ:", args, wrong_order_lifting(**args), dp_walk(**args))
        break

phản ví dụ: {'n': 6, 'parent': [-1, 4, 0, 0, 5, 2], 'gates': [[3, 1, 2], [2, 3, 3], [0, 3, 1], [3, 3, 1], [2, 2, 0], [3, 0, 1]], 'queries': [[4, 1, 2, 1], [5, 0, 3, 0], [2, 1, 5, 0], [1, 0, 0, 0]]} 93 61


### Lỗi 4 — Giả sử `parent[i] < i`

**Code sai:** duyệt `for v in range(1, n)` để tính `depth[v] = depth[parent[v]] + 1`.
**Vì sao sai:** đề chỉ nói "valid tree"; `parent[v]` có thể lớn hơn `v` nên `depth[parent[v]]` chưa được tính (= 0).
**Code đúng:** BFS từ gốc theo `children`.

### Lỗi 5 — Dùng nghịch đảo ma trận modulo

**Code sai:** tiền tố `P[v]` = tích từ gốc xuống `v`, rồi `mat(a→l) = P[a] · P[l]⁻¹`.
**Vì sao sai:** định thức có thể `≡ 0 (mod p)` (ví dụ nút `(1,1,1)`) nên không có nghịch đảo.
**Code đúng:** binary lifting hoặc segment tree để nhân đúng đoạn.

## 14. Mẹo & nhận dạng pattern

- **"Đếm số cách qua một chuỗi bước, mỗi bước chuyển trạng thái (ít trạng thái)"** → mỗi bước là một ma trận chuyển; đáp án là tích ma trận.
- **"Nhiều truy vấn trên đường đi lên cây"** → binary lifting; nếu thông tin là **tích không giao hoán**, giữ nguyên **thứ tự** khi ghép hai nửa.
- Không phải phép toán nào cũng có nghịch đảo: tổng/XOR thì có (tiền tố), `min/max` và tích ma trận modulo thì không ⇒ cần cấu trúc "đoạn" (sparse/binary lifting/segment tree).
- Với cây cho bằng `parent`, luôn dựng `children` + BFS thay vì giả sử `parent[i] < i`.
- Chia nhỏ: kiểm tra mô hình ma trận bằng ví dụ tay, rồi bằng liệt kê cổng, **trước khi** tối ưu.
- Mỗi người độc lập ⇒ nhân kết quả; chú ý trường hợp `d = 0` (đã ở LCA) cho kết quả `1`.

## 15. Biến thể

1. **Cổng có hai chiều (leo xuống, không chỉ lên):** đường đi giữa hai nút bất kỳ qua LCA; thêm ma trận chiều xuống.
2. **Màu thẻ cuối bắt buộc** (ví dụ phải là đỏ khi tới LCA): lấy `x[đỏ]` thay vì `x0 + x1` (Bài 5).
3. **Có nhiều màu thẻ** (k màu): ma trận `k×k`, độ phức tạp nhân `k^3`.
4. **Cập nhật số cổng của một nút** giữa các truy vấn: dùng segment tree trên HLD (heavy-light decomposition) thay cho bảng nhảy.
5. **Đếm đường đi tới một tổ tiên cố định (không phải LCA):** chỉ một lần leo, cùng kỹ thuật.

## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê cổng | duyệt từng cổng cụ thể | mũ | `O(D)` | kiểm chứng cây rất nhỏ |
| 2. Đi bộ từng nút | vectơ × ma trận qua từng nút | `O(q · D)` | `O(n)` | cây nông / ít truy vấn / đối chiếu |
| 3. Binary lifting | gộp trước `2^k` nút thành một ma trận | `O((n+q) log n)` | `O(n log n)` | **nộp bài** |
| 4. Offline + segment tree | giữ đường từ gốc, truy vấn đoạn | `O((n+q) log n)` | `O(n)` | khi tiết kiệm bộ nhớ, biết trước truy vấn |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** `gate_matrix([2, 3, 4])` trả về tuple nào? Và `vec_mul((0, 1), gate_matrix([2, 3, 4]))`?

**Bài 2 (tính tay).** Cây đường thẳng `0 <- 1 <- 2`, `gates[1] = [1, 2, 1]`, `gates[2] = [0, 1, 2]`. Một người ở nút `2` cầm thẻ đỏ, leo tới gốc. Có bao nhiêu cách?

**Bài 3 (điền TODO).** Hoàn thiện `my_mat_mul(A, B)` nhân hai ma trận `2×2` (tuple 4 số) modulo `10^9 + 7`.

**Bài 4 (tự viết lại).** Viết `my_ways(v, card, l, parent, gates)` đếm số cách đi từ `v` lên `l` bằng cách đi bộ từng nút.

**Bài 5 (biến thể).** Viết `ways_end_red(v, card, l, parent, gates)`: số cách mà **thẻ cuối cùng khi tới `l` là màu đỏ**.

In [21]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def my_mat_mul(A, B):
    # TODO: A = (a00, a01, a10, a11), B tương tự
    # TODO: trả về tuple 4 số (c00, c01, c10, c11) với c_ij = sum_k A_ik * B_kj, lấy % MOD
    pass

# Bài 4 — TODO
def my_ways(v, card, l, parent, gates):
    # TODO: x = (1, 0) nếu card == 0 else (0, 1); while v != l: x = x * T; v = parent[v]
    pass

# Bài 5 — TODO
def ways_end_red(v, card, l, parent, gates):
    # TODO: giống Bài 4 nhưng trả về x[1] thay vì x[0] + x[1]
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: `gate_matrix([r, b, w]) = (b, w, w, r)`.
- Mức 2: ở đây `r = 2, b = 3, w = 4`.
- Mức 3: `(3, 4, 4, 2)`. Vectơ `(0, 1) · T = (0·3 + 1·4, 0·4 + 1·2) = (4, 2)`.

**Bài 2.**
- Mức 1: qua hai nút `2` rồi `1`.
- Mức 2: nút `2` có `[0,1,2]` nên `T2 = gate_matrix([0,1,2]) = (1, 2, 2, 0)`; thẻ đỏ là `(0,1)`, nên `(0,1)·T2 = (2, 0)`.
- Mức 3: nút `1`: `T1 = gate_matrix([1,2,1]) = (2, 1, 1, 1)`; `(2,0)·T1 = (4, 2)`; tổng `6`.

**Bài 3.**
- Mức 1: công thức `c_ij = a_i0·b_0j + a_i1·b_1j`.
- Mức 2: tuple `(a00,a01,a10,a11)` nên `A[0], A[1]` là hàng 0, `A[2], A[3]` là hàng 1.
- Mức 3: `c00 = A[0]*B[0] + A[1]*B[2]`.

**Bài 4.**
- Mức 1: dùng `gate_matrix` và `vec_mul` ở phần 3.
- Mức 2: vòng `while v != l`.
- Mức 3: `x = vec_mul(x, gate_matrix(gates[v])); v = parent[v]`.

**Bài 5.**
- Mức 1: màu đỏ có chỉ số `1`.
- Mức 2: chỉ cần phần tử đỏ của vectơ cuối.
- Mức 3: `return x[1] % MOD`.

## 19. Đáp án tham khảo

In [22]:
# Bài 1
assert gate_matrix([2, 3, 4]) == (3, 4, 4, 2)
assert vec_mul((0, 1), gate_matrix([2, 3, 4])) == (4, 2)

# Bài 2
parent2 = [-1, 0, 1]
gates2 = [[0, 0, 0], [1, 2, 1], [0, 1, 2]]
assert ways_walk(2, 1, 0, parent2, gates2) == 6
assert ways_enum(2, 1, 0, parent2, gates2) == 6

# Bài 3
def my_mat_mul_ans(A, B):
    return ((A[0] * B[0] + A[1] * B[2]) % MOD, (A[0] * B[1] + A[1] * B[3]) % MOD,
            (A[2] * B[0] + A[3] * B[2]) % MOD, (A[2] * B[1] + A[3] * B[3]) % MOD)

rng = random.Random(0)
for _ in range(200):
    A = tuple(rng.randrange(MOD) for _ in range(4)); B = tuple(rng.randrange(MOD) for _ in range(4))
    assert my_mat_mul_ans(A, B) == mat_mul(A, B)
print("Bài 1–3 OK")

Bài 1–3 OK


In [23]:
# Bài 4
def my_ways_ans(v, card, l, parent, gates):
    x = (1, 0) if card == 0 else (0, 1)
    while v != l:
        x = vec_mul(x, gate_matrix(gates[v]))
        v = parent[v]
    return (x[0] + x[1]) % MOD

# Bài 5: thẻ cuối cùng là đỏ
def ways_end_red_ans(v, card, l, parent, gates):
    x = (1, 0) if card == 0 else (0, 1)
    while v != l:
        x = vec_mul(x, gate_matrix(gates[v]))
        v = parent[v]
    return x[1] % MOD

def brute_end_red(v, card, l, parent, gates):
    def go(u, c):
        if u == l:
            return 1 if c == 1 else 0
        r, b, w = gates[u]
        tot = 0
        for kind in ["R"] * r + ["B"] * b + ["W"] * w:
            if kind == "R" and c == 1: tot += go(parent[u], 1)
            elif kind == "B" and c == 0: tot += go(parent[u], 0)
            elif kind == "W": tot += go(parent[u], 1 - c)
        return tot
    return go(v, card)

rng = random.Random(21)
for _ in range(300):
    n = rng.randint(2, 7)
    parent = random_tree(rng, n)
    gates = [[rng.randint(0, 3) for _ in range(3)] for _ in range(n)]
    v, card = rng.randrange(n), rng.randint(0, 1)
    l = 0                                        # gốc là tổ tiên của mọi nút
    assert my_ways_ans(v, card, l, parent, gates) == ways_enum(v, card, l, parent, gates)
    assert ways_end_red_ans(v, card, l, parent, gates) == brute_end_red(v, card, l, parent, gates)
print("Bài 4–5 OK")

Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template binary lifting với tích không giao hoán:**

```python
# anc[0][v] = cha; mat[0][v] = ma trận của nút v
for k in range(1, LOG):
    for v in range(n):
        mid = anc[k-1][v]
        anc[k][v] = anc[k-1][mid]
        mat[k][v] = mat[k-1][v] @ mat[k-1][mid]      # thứ tự: nửa gần v trước, nửa xa sau

def climb(v, x, d):                                  # x là vectơ hàng
    for k in range(LOG):
        if d >> k & 1:
            x = x @ mat[k][v]; v = anc[k][v]
    return x
```

**Checklist trước khi submit:**

- [ ] Ma trận `T = [[b, w], [w, r]]` với chỉ số `0 = xanh`, `1 = đỏ` khớp với mã hóa thẻ của đề (`1 = đỏ`, `0 = xanh`).
- [ ] Nhân theo thứ tự từ nút xuất phát lên trên (vectơ hàng nhân từ trái).
- [ ] Số cách của người đã ở LCA là `1` (`d = 0`).
- [ ] Dựng `depth` bằng BFS, không giả sử `parent[i] < i`.
- [ ] Gốc trỏ cha về chính nó trong `anc[0]` để tránh chỉ số `-1`.
- [ ] `mod` sau mỗi phép nhân; XOR sau khi đã `mod` từng truy vấn.
- [ ] Tên phương thức bài Premium cần đối chiếu lại với trình soạn thảo thật.
- [ ] Đã đối chiếu liệt kê cổng trên cây nhỏ ngẫu nhiên.